<a href="https://colab.research.google.com/github/helenjoy/tiny-ml/blob/main/quantization_in_tiny_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Teaching quantization in TinyML requires bridging abstract mathematical concepts with practical hardware constraints. Here is a battle-tested curriculum structure designed to take students from core concepts to working code.

**1. Core Definition & The "Why"**

Start with a clear physical analogy before diving into math:

* **The Analogy:** High-precision model weights are like storing water in ultra-fine 32-bit glass vials (`float32`). Quantization is pouring that water into simple 8-bit marked measuring cups (`int8`). You lose a tiny fraction of precision, but the cups are far cheaper, smaller, and easier to transport.
* **The Core Formula:** Explain the linear mapping function used to convert floating-point values ($q$) to integers ($x$):

$$x = \text{round}\left(\frac{q}{S}\right) + Z$$



Where $S$ is the **Scale** (a floating-point factor) and $Z$ is the **Zero-point** (an integer representing real zero).

---

**2. The 3 Main Types of Quantization**

Break down the trade-offs using a clear comparison matrix:

| Quantization Type | How It Works | Calibration Data Needed? | Target Hardware | Ease of Implementation |
| --- | --- | --- | --- | --- |
| **Dynamic Range** | Quantizes weights to `int8`; keeps activations as `float32` until runtime. | No | Mobile CPUs (iOS / Android) | Extremely Easy |
| **Full Integer** | Quantizes both weights and activations to `int8`. Requires calibration. | Yes (Representative Dataset) | Microcontrollers (Cortex-M), Edge TPUs | Moderate |
| **Quantization-Aware Training (QAT)** | Emulates quantization math *during* model training so weights adapt to precision loss. | Yes (Full Retraining) | Ultra-constrained devices requiring max accuracy | Advanced |

---

**3. Code Implementation Flow (TensorFlow Lite)**

Guide students through writing the code step-by-step using standard python structures.

**Step 1: Train/Load a Baseline Model**

In [ ]:
# ==============================================================================
# TinyML Quantization Lab: From Training to Microcontroller Optimization
# ==============================================================================

import os
import time
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

print(f"TensorFlow Version: {tf.__version__}")

# ------------------------------------------------------------------------------
# STEP 0: Prepare a Dataset & Train a Baseline Model (CIFAR-10 Subsample)
# ------------------------------------------------------------------------------
print("\n--- Step 0: Loading Data & Training Baseline Model ---")

# Load CIFAR-10 data
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalize pixel values to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Build a compact CNN model
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(16, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(10, activation="softmax"),
    ]
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Train for 1 quick epoch for lab demonstration
model.fit(
    x_train[:5000],
    y_train[:5000],
    epochs=1,
    batch_size=64,
    validation_split=0.1,
)

# Save baseline unquantized float32 model
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_float_model = converter.convert()

with open("model_float32.tflite", "wb") as f:
    f.write(tflite_float_model)

print("Baseline float32 model created.")

# ------------------------------------------------------------------------------
# STEP 1: Dynamic Range Quantization
# ------------------------------------------------------------------------------
print("\n--- Step 1: Generating Dynamic Range Quantized Model ---")

converter_dynamic = tf.lite.TFLiteConverter.from_keras_model(model)
converter_dynamic.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_dynamic_model = converter_dynamic.convert()

with open("model_dynamic.tflite", "wb") as f:
    f.write(tflite_dynamic_model)

print("Dynamic range model created.")

# ------------------------------------------------------------------------------
# STEP 2: Full Integer Quantization (Full INT8)
# ------------------------------------------------------------------------------
print("\n--- Step 2: Generating Full INT8 Quantized Model ---")


# Define representative dataset generator using training samples
def representative_data_gen():
    # Take 100 samples from training set for calibration
    for i in range(100):
        # Add batch dimension: (1, 32, 32, 3)
        sample = np.expand_dims(x_train[i], axis=0).astype(np.float32)
        yield [sample]


converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_data_gen

# Strictly enforce INT8 execution for Microcontrollers (TFLM)
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type = tf.int8
converter_int8.inference_output_type = tf.int8

tflite_int8_model = converter_int8.convert()

with open("model_full_int8.tflite", "wb") as f:
    f.write(tflite_int8_model)

print("Full INT8 model created successfully.")

# ------------------------------------------------------------------------------
# STEP 3: Verification Lab (Size, Accuracy, and Latency Evaluation)
# ------------------------------------------------------------------------------
print("\n--- Step 3: Hands-On Verification Lab Results ---")


# 3A. File Size Comparison
def get_file_size_kb(filepath):
    return os.path.getsize(filepath) / 1024.0


size_float = get_file_size_kb("model_float32.tflite")
size_dynamic = get_file_size_kb("model_dynamic.tflite")
size_int8 = get_file_size_kb("model_full_int8.tflite")


# 3B. Inference & Accuracy Evaluation Function
def evaluate_model(model_path, x_data, y_data, is_int8=False):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    correct_predictions = 0
    total_samples = len(x_data)

    start_time = time.time()

    for i in range(total_samples):
        input_data = np.expand_dims(x_data[i], axis=0)

        # Handle INT8 Quantization Scaling
        if is_int8:
            scale, zero_point = input_details["quantization"]
            input_data = (input_data / scale + zero_point).astype(np.int8)

        interpreter.set_tensor(input_details["index"], input_data)
        interpreter.invoke()

        output = interpreter.get_tensor(output_details["index"])

        if is_int8:
            prediction = np.argmax(output[0])
        else:
            prediction = np.argmax(output[0])

        if prediction == y_data[i][0]:
            correct_predictions += 1

    total_time = time.time() - start_time
    avg_latency_ms = (total_time / total_samples) * 1000
    accuracy = correct_predictions / total_samples

    return accuracy, avg_latency_ms


# Evaluate on 500 test images for speed
eval_samples = 500
acc_float, latency_float = evaluate_model(
    "model_float32.tflite", x_test[:eval_samples], y_test[:eval_samples]
)
acc_dynamic, latency_dynamic = evaluate_model(
    "model_dynamic.tflite", x_test[:eval_samples], y_test[:eval_samples]
)
acc_int8, latency_int8 = evaluate_model(
    "model_full_int8.tflite",
    x_test[:eval_samples],
    y_test[:eval_samples],
    is_int8=True,
)

# ------------------------------------------------------------------------------
# STEP 4: Output Comparison Report
# ------------------------------------------------------------------------------
print("\n" + "=" * 65)
print(f"{'Model Version':<20} | {'Size (KB)':<10} | {'Accuracy':<10} | {'Latency (ms)'}")
print("=" * 65)
print(
    f"{'Float32 Baseline':<20} | {size_float:<10.2f} | {acc_float*100:<9.2f}% | {latency_float:.3f} ms"
)
print(
    f"{'Dynamic Range':<20} | {size_dynamic:<10.2f} | {acc_dynamic*100:<9.2f}% | {latency_dynamic:.3f} ms"
)
print(
    f"{'Full INT8':<20} | {size_int8:<10.2f} | {acc_int8*100:<9.2f}% | {latency_int8:.3f} ms"
)
print("=" * 65)

# ------------------------------------------------------------------------------
# STEP 5: Export full_int8 model for Microcontrollers (C++ Array Header)
# ------------------------------------------------------------------------------
!xxd -i model_full_int8.tflite > model_data.h
print("\nGenerated 'model_data.h' for C++/Arduino MCU deployment.")

TensorFlow Version: 2.20.0

--- Step 0: Loading Data & Training Baseline Model ---
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1747s 10us/step
71/71 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - accuracy: 0.2278 - loss: 2.0919 - val_accuracy: 0.3000 - val_loss: 1.8688
Saved artifact at '/tmp/tmpidtex0mj'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 3), dtype=tf.float32, name='keras_tensor_10')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  134656846001936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134656846000592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134656846000016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134656727325520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134656727325328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134656727323600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134656727318608: TensorSpec(shape=()

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Full INT8 model created successfully.

--- Step 3: Hands-On Verification Lab Results ---

Model Version        | Size (KB)  | Accuracy   | Latency (ms)
Float32 Baseline     | 314.73     | 32.00    % | 0.096 ms
Dynamic Range        | 86.29      | 32.00    % | 0.095 ms
Full INT8            | 85.30      | 32.00    % | 0.155 ms
/bin/bash: line 1: xxd: command not found

Generated 'model_data.h' for C++/Arduino MCU deployment.


**Step 2: Apply Dynamic Range Quantization**

**Step 3: Apply Full Integer Quantization**

---

**4. Hands-on Lab & Verification**

Have students run a three-part validation check to see the real-world impact:

* **Size Verification:** Compare file sizes on disk using `os.path.getsize()`. Expect a **~4x reduction** (e.g., 12MB down to 3MB).
* **Accuracy Check:** Run validation sets against both baseline and quantized models to evaluate top-1 accuracy degradation (usually $<1\%$).
* **Latency Benchmark:** Test deployment latency on actual hardware (e.g., ESP32, Arduino Nano 33 BLE, or Raspberry Pi Pico) using TensorFlow Lite for Microcontrollers (TFLM).